## Analysis of the splits of the dataset

Extract the data from QCArchive

In [23]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

import deepchem as dc
import numpy as np

from rdkit.DataStructs.cDataStructs import BulkTanimotoSimilarity


def smiles_to_fps(smiles, radius=2, nbits=1024):
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=nbits)
           for m in mols if m is not None]
    return fps

# use metadata from models
TRAIN_FOLDER = "./maxmin-train/"
TEST_FOLDER = "./maxmin-test/"
train_dataset = dc.data.DiskDataset(TRAIN_FOLDER)
test_dataset = dc.data.DiskDataset(TEST_FOLDER)


In [24]:
smiles_in_train = list(set(train_dataset.ids))
smiles_in_test = list(set(test_dataset.ids))

fps_a = smiles_to_fps(smiles_in_train)
fps_b = smiles_to_fps(smiles_in_test)

arr_a = np.asarray([np.frombuffer(fp.ToBitString().encode('utf-8'), 'S1') == b'1' for fp in fps_a])
arr_b = np.asarray([np.frombuffer(fp.ToBitString().encode('utf-8'), 'S1') == b'1' for fp in fps_b])

max_sim, min_sim = 0, 1
max_pair, min_pair = None, None

for i, fp in enumerate(fps_a):
    sims = BulkTanimotoSimilarity(fp, fps_b)
    j_max = np.argmax(sims)
    j_min = np.argmin(sims)
    if sims[j_max] > max_sim:
        max_sim = sims[j_max]
        max_pair = (smiles_in_train[i], smiles_in_test[j_max])
    if sims[j_min] < min_sim:
        min_sim = sims[j_min]
        min_pair = (smiles_in_train[i], smiles_in_test[j_min])



[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerator
[16:52:01] DEPRECATION WARNING: please use MorganGenerat

In [25]:
max_pair

('[H:12][C:1]([H:13])([H:14])[C:2]([H:15])([H:16])[C:3]([H:17])([H:18])[C:4]([H:19])([H:20])[C:5]([H:21])([H:22])[N+:6]([H:23])([C:7]([H:24])([H:25])[C:8]([H:26])([H:27])[H:28])[C:9]([H:29])([H:30])[C:10]([H:31])([H:32])[O:11][H:33]',
 '[H:12][C:1]([H:13])([H:14])[C:2]([H:15])([H:16])[C:3]([H:17])([H:18])[C:4]([H:19])([H:20])[C:5]([H:21])([H:22])[N+:6]([H:23])([C:7]([H:24])([H:25])[C:8]([H:26])([H:27])[H:28])[C:9]([H:29])([H:30])[C:10]([H:31])([H:32])[O:11][H:33]')